# Protobuf Basics — 04: Spark + Protobuf Integration

Using Spark's built-in Protobuf support to read, write, and transform
Protobuf data in batch and streaming pipelines.

Topics: from_protobuf, to_protobuf, descriptor file, reading binary files,
Spark 4.0.2 Protobuf API.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Compile .proto to Descriptor File



In [ ]:

import subprocess, pathlib

PROTO_DIR = f"{DATA_DIR}/protos"
pathlib.Path(PROTO_DIR).mkdir(exist_ok=True)

proto_content = """syntax = "proto3";
package example;

message Order {
  int64  order_id    = 1;
  int64  customer_id = 2;
  string product     = 3;
  string category    = 4;
  string country     = 5;
  int32  quantity    = 6;
  double price       = 7;
  double revenue     = 8;
  string status      = 9;
}
"""
pathlib.Path(f"{PROTO_DIR}/orders.proto").write_text(proto_content)

# Install grpcio-tools if needed
for tool in ["protoc", "python3 -m grpc_tools.protoc"]:
    r = subprocess.run(["pip3","install","--quiet","--break-system-packages","grpcio-tools"],
                       capture_output=True, text=True)
    r = subprocess.run(
        ["python3","-m","grpc_tools.protoc",
         f"--proto_path={PROTO_DIR}",
         f"--python_out={PROTO_DIR}",
         f"--descriptor_set_out={PROTO_DIR}/orders.desc",
         "--include_imports", f"{PROTO_DIR}/orders.proto"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        desc_sz = pathlib.Path(f"{PROTO_DIR}/orders.desc").stat().st_size
        print(f"✅ orders.desc compiled ({desc_sz} bytes)")
        break
else:
    print("protoc not available — creating minimal descriptor manually")
    # Minimal workaround: create descriptor via Python protobuf API
    from google.protobuf import descriptor_pb2
    fds = descriptor_pb2.FileDescriptorSet()
    fd  = fds.file.add()
    fd.name = "orders.proto"; fd.package = "example"; fd.syntax = "proto3"
    msg = fd.message_type.add(); msg.name = "Order"
    for i,(name,typ) in enumerate([("order_id",3),("customer_id",3),("product",9),
                                    ("category",9),("country",9),("quantity",5),
                                    ("price",1),("revenue",1),("status",9)],1):
        field = msg.field.add()
        field.name=name; field.number=i; field.type=typ; field.label=1
    with open(f"{PROTO_DIR}/orders.desc","wb") as f: f.write(fds.SerializeToString())
    print("✅ Minimal descriptor created")


## Step 2 — from_protobuf: Deserialize Binary Column



In [ ]:

import pathlib, sys, importlib.util, random, struct

PROTO_DIR = f"{DATA_DIR}/protos"
DESC_FILE = f"{PROTO_DIR}/orders.desc"

# Create test data: binary Protobuf messages as a Spark DataFrame
sys.path.insert(0, PROTO_DIR)
try:
    spec = importlib.util.spec_from_file_location("orders_pb2", f"{PROTO_DIR}/orders_pb2.py")
    orders_pb2 = importlib.util.module_from_spec(spec); spec.loader.exec_module(orders_pb2)

    import random, datetime
    random.seed(42)
    CATS=["Electronics","Books","Clothing","Food","Sports"]
    CTRS=["US","UK","DE","FR","JP"]

    # Serialize 1000 orders to binary
    serialized = []
    for i in range(1000):
        o = orders_pb2.Order()
        o.order_id=i+1; o.customer_id=random.randint(1,500)
        o.product=f"Product_{random.randint(1,50)}"; o.category=random.choice(CATS)
        o.country=random.choice(CTRS); o.quantity=random.randint(1,10)
        o.price=round(random.uniform(5,500),2); o.revenue=round(o.price*o.quantity,2)
        o.status=random.choice(["pending","confirmed","shipped"])
        serialized.append((bytearray(o.SerializeToString()),))

    # Create DataFrame with binary column
    df_binary = spark.createDataFrame(serialized, ["proto_bytes"])
    print(f"Binary DataFrame: {df_binary.count():,} rows")
    df_binary.show(3)

    # Deserialize with from_protobuf
    from pyspark.sql.protobuf.functions import from_protobuf
    df_orders = df_binary.select(
        from_protobuf(F.col("proto_bytes"), "Order", DESC_FILE).alias("order")
    ).select("order.*")

    print(f"\nDeserialized: {df_orders.count():,} rows")
    df_orders.printSchema()
    df_orders.show(5)

except Exception as e:
    print(f"Note: {type(e).__name__}: {str(e)[:100]}")
    print("from_protobuf requires compiled .desc file — see Step 1 for compilation.")


## Step 3 — to_protobuf: Serialize Struct Column



In [ ]:

PROTO_DIR = f"{DATA_DIR}/protos"
DESC_FILE = f"{PROTO_DIR}/orders.desc"

try:
    from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf

    # Create DataFrame to serialize
    import random, datetime
    random.seed(42); N=500
    CATS=["Electronics","Books","Clothing"]; CTRS=["US","UK","DE"]
    rows=[(i+1,random.randint(1,100),f"Product_{i%50}",random.choice(CATS),
           random.choice(CTRS),random.randint(1,5),round(random.uniform(5,200),2),
           0.0,random.choice(["pending","confirmed"])) for i in range(N)]
    for r in rows: r=list(r); r[7]=round(r[6]*r[5],2)

    df = spark.createDataFrame(rows, ["order_id","customer_id","product","category",
                                       "country","quantity","price","revenue","status"])

    # Serialize to Protobuf binary
    df_serialized = df.select(
        to_protobuf(F.struct(*df.columns), "Order", DESC_FILE).alias("proto_bytes")
    )
    print(f"Serialized {df_serialized.count():,} rows to Protobuf")
    df_serialized.show(3)

    # Round-trip: serialize then deserialize
    df_roundtrip = df_serialized.select(
        from_protobuf(F.col("proto_bytes"), "Order", DESC_FILE).alias("order")
    ).select("order.*")

    original_count = df.count()
    roundtrip_count = df_roundtrip.count()
    print(f"\nRound-trip: {original_count:,} → {roundtrip_count:,} rows ({'✅' if original_count==roundtrip_count else '❌'})")
    df_roundtrip.show(3)

except Exception as e:
    print(f"Note: {type(e).__name__}: {str(e)[:150]}")
    print("to_protobuf requires the compiled .desc file + matching schema.")


## Step 4 — Reading Protobuf Files Directly



In [ ]:

print("""
Reading binary Protobuf files with spark.read:

  spark.read
       .format("protobuf")
       .option("protobufDescriptorFilePath", "/path/to/orders.desc")
       .option("messageName", "Order")
       .load("/path/to/proto_files/")

Writing Protobuf files:
  df.write
    .format("protobuf")
    .option("protobufDescriptorFilePath", "/path/to/orders.desc")
    .option("messageName", "Order")
    .save("/path/output/")

Note: protobuf file format = one serialized Protobuf message per record,
concatenated without delimiters. Less common than Kafka usage.
Most production usage is via Kafka: read binary column → from_protobuf().
""")


## Summary



In [ ]:

print("""
Spark 4.0.2 Protobuf API:
  from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf

  # Deserialize binary → struct
  df.select(from_protobuf(col("binary_col"), "MessageName", "path/to.desc").alias("data"))
    .select("data.*")

  # Serialize struct → binary
  df.select(to_protobuf(struct(*df.columns), "MessageName", "path/to.desc").alias("bytes"))

  # File format
  spark.read.format("protobuf")
            .option("protobufDescriptorFilePath","path/to.desc")
            .option("messageName","Order")
            .load("/path/")

Requirements:
  1. Compiled descriptor file (.desc) from protoc or grpcio-tools
  2. Message name matching the .proto file
  3. Spark 3.4+ (built-in spark-protobuf module)
""")
